# ChaCha20 Stream Cipher: Implementation Skeleton in SageMath

This notebook provides a structured implementation of the ChaCha20 stream cipher from scratch using Python/SageMath.
Each function includes a self-descriptive docstring. Unit & Integration tests, validated with `assert` statements, are provided in an additional notebook.

> **Tip:** ChaCha20 is built entirely from 32-bit integer additions, XOR (`^^` in SageMath), and bitwise left-rotations. No tables, no S-boxes: just ARX (Add-Rotate-XOR).

**Reference:** [RFC 8439 - ChaCha20 and Poly1305 for IETF Protocols](https://datatracker.ietf.org/doc/html/rfc8439)

## 1. Algorithm Overview

ChaCha20 is a **stream cipher** designed by Daniel J. Bernstein. Unlike DES/AES, it does *not* operate on fixed-size blocks of plaintext; instead, it generates a **pseudorandom keystream** of any length, which is then XOR-ed with the plaintext.

The cipher state is a **4x4 matrix of 32-bit words** (512 bits total):

<div style="font-family: monospace; white-space: pre; line-height: 1.8; margin: 16px 0;">
 Word index:  [ 0]  [ 1]  [ 2]  [ 3]
              [ 4]  [ 5]  [ 6]  [ 7]
              [ 8]  [ 9]  [10]  [11]
              [12]  [13]  [14]  [15]

 Semantics:   [C0]  [C1]  [C2]  [C3]   &lt;- 128-bit constant ("expand 32-byte k")
              [K0]  [K1]  [K2]  [K3]   &lt;- key words  0-3  (bits   0-127)
              [K4]  [K5]  [K6]  [K7]   &lt;- key words  4-7  (bits 128-255)
              [CTR] [N0]  [N1]  [N2]   &lt;- 32-bit counter + 96-bit nonce
</div>

The **keystream generation** for one 64-byte block:

<div style="font-family: monospace; white-space: pre; line-height: 1.5; margin: 16px 0;">
  key (32 bytes) --+
nonce (12 bytes) --+--&gt; chacha20_init_state()  --&gt;  initial state (16 x u32)
   counter (u32) --+                                      |
                                                          |  working_state = copy(initial_state)
                                                          |
                                                          v
                                                 +------------------+
                                                 |  double_round()  | x 10
                                                 | (column + diag)  |
                                                 +--------+---------+
                                                          |
                                         working_state[i] += initial_state[i]  (mod 2^32)
                                                          |
                                                          v
                                              serialize to 64 bytes (little-endian)
                                                          |
                                                          v
                                         keystream[0:64]  XOR  plaintext[0:64]
</div>

### The Quarter Round function: core primitive

The **quarter round** `QR(a, b, c, d)` operates on four 32-bit words and mixes them using four ARX steps. All additions are modulo 2^32. This is also called carryless addition on a 32-bit word:

```
a = a + b (mod 32);  d = d XOR a;  d = d ROTL 16
c = c + d (mod 32);  b = b XOR c;  b = b ROTL 12
a = a + b (mod 32);  d = d XOR a;  d = d ROTL 8
c = c + d (mod 32);  b = b XOR c;  b = b ROTL 7
```

### Column and Diagonal Rounds

Each **double round** applies the quarter round to columns first, then to diagonals of the 4x4 state:

| Round type | Applications |
|---|---|
| **Column round** | QR(s[0],s[4],s[8],s[12])  /  QR(s[1],s[5],s[9],s[13])  /  QR(s[2],s[6],s[10],s[14])  /  QR(s[3],s[7],s[11],s[15]) |
| **Diagonal round** | QR(s[0],s[5],s[10],s[15])  /  QR(s[1],s[6],s[11],s[12])  /  QR(s[2],s[7],s[8],s[13])  /  QR(s[3],s[4],s[9],s[14]) |

ChaCha**20** applies **10 double rounds** (= 20 total quarter-round passes).

### Constants

ChaCha20 has no secret tables. The only constants are the four 32-bit words that fill row 0 of the state matrix (the "nothing-up-my-sleeve" magic string). The come from the ASCII string `expand 32-byte k` decoded as four little-endian 32-bit words.

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:12px 16px; border-radius:4px; font-family:sans-serif;">

<b>Important: Little-endian byte order</b>
<br><br>
ChaCha20 treats <b>all multi-byte values as little-endian</b> (RFC 8439 §2.1). This is the opposite of the big-endian (MSB-first) network byte order you may be used to.

When loading a 32-bit word from four bytes <code>[b0, b1, b2, b3]</code>:
<pre style="background:#fff3cd; padding:4px;">
word = b0 | (b1 &lt;&lt; 8) | (b2 &lt;&lt; 16) | (b3 &lt;&lt; 24)   # little-endian
     = int.from_bytes([b0, b1, b2, b3], "little")
</pre>

Python's <code>int.from_bytes(data, "little")</code> and <code>n.to_bytes(4, "little")</code> handle this transparently.
<br><br>
<b>Consequence:</b> the key bytes <code>00 01 02 03</code> load as the word <code>0x03020100</code>, <i>not</i> <code>0x00010203</code>.
This appears throughout all test vectors in RFC 8439.

</div>

<div style="background:#f8d7da; border-left:4px solid #dc3545; padding:12px 16px; border-radius:4px; font-family:sans-serif;">

<b>CRITICAL: XOR in SageMath kernels</b>
<br><br>
In a <b>SageMath</b> kernel, the caret operator <code>^</code> is <b>exponentiation</b>, not bitwise XOR.
<br><br>
<b>Always use <code>^^</code> for bitwise XOR in SageMath.</b>
<pre style="background:#f8d7da; padding:4px;">
# Wrong in SageMath -- this computes a^b (power), not XOR!
result = a ^ b

# Correct in SageMath
result = a ^^ b
</pre>
Since ChaCha20 XORs 32-bit words, using <code>^</code> by mistake will silently produce an astronomically large integer instead of a bitwise result.

</div>

## 2. Implementation

### Part A - State Initialization: `chacha20_init_state` and `serialize_state`

Before any mixing can take place, the 16-word state matrix must be constructed from the inputs:

```
  state[0..3]   = MAGIC_CONSTANTS         (4 words, fixed, see "Constants" in overview section)
  state[4..11]  = key                     (8 words, little-endian u32)
  state[12]     = counter                 (1 word,  32-bit integer)
  state[13..15] = nonce                   (3 words, little-endian u32)
```

`chacha20_init_state` packs the raw `bytes` inputs into a flat Python list of 16 integers.

`serialize_state` does the inverse after mixing: it converts each word back to 4 bytes (little-endian) and concatenates all 16 words into a 64-byte keystream block.

> **Hint for `chacha20_init_state`:** Use `int.from_bytes(..., "little")` to each 32-bit word from a byte string.

> **Hint for `serialize_state`:** You can use `b"".join()` or `struct.pack()` (more elegant).

In [ ]:
def chacha20_init_state(key: bytes, counter: int, nonce: bytes) -> list[int]:
    """
    Build the initial 16-word ChaCha20 state matrix.

    All multi-byte values are loaded as little-endian 32-bit words (RFC 8439 ss.2.3).

    Parameters
    ----------
    key     : 32-byte (256-bit) secret key.
    counter : 32-bit block counter (starts at 0 or 1, incremented per block).
    nonce   : 12-byte (96-bit) nonce (must never be reused with the same key).

    Returns
    -------
    List of 16 integers, each a 32-bit word, representing the flat state.

    Layout
    ------
    state[0..3]   = MAGIC_CONSTANTS
    state[4..11]  = key words (8 x 32-bit little-endian)
    state[12]     = counter
    state[13..15] = nonce words (3 x 32-bit little-endian)
    """
    assert len(key) == 32, f"Key must be 32 bytes, got {len(key)}"
    assert len(nonce) == 12, f"Nonce must be 12 bytes, got {len(nonce)}"
    pass  # TODO


def serialize_state(state: list[int]) -> bytes:
    """
    Serialize a 16-word state list into a 64-byte little-endian byte string.

    This is the final step of chacha20_block(): it converts each 32-bit word
    back to 4 bytes (little-endian) and concatenates the results.

    Parameters
    ----------
    state : List of 16 integers, each a 32-bit word.

    Returns
    -------
    64-byte keystream block.
    """
    pass  # TODO


# -- Tests ---------------------------------------------------------------------

# State layout: constants must occupy words 0-3
_test_key = bytes(range(32))  # 00 01 02 ... 1f
_test_nonce = bytes([0] * 12)
_test_state = chacha20_init_state(_test_key, 0, _test_nonce)

assert len(_test_state) == 16, "State must have exactly 16 words"
assert _test_state[0] == 0x61707865, "Constant word 0 wrong"
assert _test_state[1] == 0x3320646E, "Constant word 1 wrong"
assert _test_state[2] == 0x79622D32, "Constant word 2 wrong"
assert _test_state[3] == 0x6B206574, "Constant word 3 wrong"
# Key bytes 00 01 02 03 load as little-endian word 0x03020100
assert _test_state[4] == 0x03020100, "Key word 0 wrong (check endianness)"
assert _test_state[12] == 0, "Counter must be 0"

# serialize_state round-trip
_serialized = serialize_state(_test_state)
assert len(_serialized) == 64, "Serialized block must be 64 bytes"
assert int.from_bytes(_serialized[0:4], "little") == 0x61707865, (
    "First serialized word wrong"
)
assert int.from_bytes(_serialized[16:20], "little") == 0x03020100, (
    "Key word 0 wrong after serialize"
)

# RFC 8439 ss.2.3.2 -- nonce and counter placement
_key2 = bytes(range(32))
_nonce2 = bytes.fromhex("000000090000004a00000000")
_s2 = chacha20_init_state(_key2, 1, _nonce2)
assert _s2[12] == 1, "Counter word must equal the counter argument"
assert _s2[13] == 0x09000000, "Nonce word 0 wrong (little-endian!)"
assert _s2[14] == 0x4A000000, "Nonce word 1 wrong (little-endian!)"
assert _s2[15] == 0x00000000, "Nonce word 2 wrong"

print("Part A -- All tests passed.")

### Part B - Core ARX Primitives: `rotate_left_32` and `quarter_round`

This is the lowest-level building block of ChaCha20. Every operation in the cipher reduces to two primitives:

- **`rotate_left_32(value, n)`** is the circular left-shift of a 32-bit word by `n` positions. Unlike DES (which has variable-width rotations), ChaCha20 always operates on 32-bit words.
- **`quarter_round(a, b, c, d)`** takes four 32-bit words and returns four new ones after the four ARX steps shown in the overview. Returns a tuple `(a, b, c, d)`.

> **Hint for `rotate_left_32`:** A 32-bit left rotation by `n` can be written as
> `((value << n) | (value >> (32 - n))) & 0xffffffff`
> The mask `& 0xffffffff` is essential because Python integers have arbitrary precision.

> **Hint for `quarter_round`:** Additions are `mod 2^32`, apply `& 0xffffffff` after every `+`. Use `^^` (SageMath) for XOR. Use `rotate_left_32` for the rotation steps.

In [ ]:
def rotate_left_32(value: int, n: int) -> int:
    """
    Circular left-rotation of a 32-bit word by n positions.

    All ChaCha20 rotations are 32-bit; the width is fixed and does not need
    to be passed as a parameter (unlike the DES rotate_left helper).

    Example: rotate_left_32(0x00000001, 1)  == 0x00000002
             rotate_left_32(0x80000000, 1)  == 0x00000001   (MSB wraps to LSB)
             rotate_left_32(0x12345678, 16) == 0x56781234

    Parameters
    ----------
    value : 32-bit input word (Python integer).
    n     : Number of positions to rotate left (0 <= n < 32).

    Returns
    -------
    32-bit rotated word.
    """
    pass  # TODO


def quarter_round(a: int, b: int, c: int, d: int) -> tuple[int, int, int, int]:
    """
    Apply the ChaCha20 quarter round to four 32-bit words.

    The four ARX steps (all additions mod 2^32, XOR with ^^, rotate with rotate_left_32):
        a = (a + b) & 0xffffffff;  d ^^= a;  d = rotate_left_32(d, 16)
        c = (c + d) & 0xffffffff;  b ^^= c;  b = rotate_left_32(b, 12)
        a = (a + b) & 0xffffffff;  d ^^= a;  d = rotate_left_32(d,  8)
        c = (c + d) & 0xffffffff;  b ^^= c;  b = rotate_left_32(b,  7)

    Parameters
    ----------
    a, b, c, d : Four 32-bit input words (Python integers).

    Returns
    -------
    Tuple (a, b, c, d) of four 32-bit output words.
    """
    # CRITICAL: Use ^^ for XOR in SageMath, NOT ^
    pass  # TODO


# -- Tests (RFC 8439 ss.2.1.1) -------------------------------------------------
# fmt:off

# Rotation sanity checks
assert rotate_left_32(0x00000001, 1) == 0x00000002
assert rotate_left_32(0x80000000, 1) == 0x00000001, "MSB must wrap to bit 0"
assert rotate_left_32(0x12345678, 16) == 0x56781234
assert rotate_left_32(0xffffffff, 8) == 0xffffffff, "All-ones is rotation-invariant"

# Quarter round -- official RFC 8439 ss.2.1.1 test vector
a, b, c, d = quarter_round(0x11111111, 0x01020304, 0x9b8d6f43, 0x01234567)
assert a == 0xea2a92f4, f"a wrong: {hex(a)}"
assert b == 0xcb1cf8ce, f"b wrong: {hex(b)}"
assert c == 0x4581472e, f"c wrong: {hex(c)}"
assert d == 0x5881c4bb, f"d wrong: {hex(d)}"

# fmt:on

print("Part B -- All tests passed.")

### Part C - Block Function: `double_round` and `chacha20_block`

This is the **cryptographic core** of ChaCha20. The block function takes the initial state, applies 20 rounds of mixing, and produces a 64-byte keystream block.

`double_round(state)` applies one column round followed by one diagonal round to the **mutable** state list (modify in place and return). The column and diagonal indices are fixed:

| Step | Quarter round call |
|---|---|
| Column 0 | `QR(state[0],  state[4],  state[8],  state[12])` |
| Column 1 | `QR(state[1],  state[5],  state[9],  state[13])` |
| Column 2 | `QR(state[2],  state[6],  state[10], state[14])` |
| Column 3 | `QR(state[3],  state[7],  state[11], state[15])` |
| Diagonal 0 | `QR(state[0],  state[5],  state[10], state[15])` |
| Diagonal 1 | `QR(state[1],  state[6],  state[11], state[12])` |
| Diagonal 2 | `QR(state[2],  state[7],  state[8],  state[13])` |
| Diagonal 3 | `QR(state[3],  state[4],  state[9],  state[14])` |

`chacha20_block` then:
1. Calls `chacha20_init_state` to get the initial state.
2. Makes a **working copy** of the state.
3. Calls `double_round` **10 times** on the working copy (= 20 rounds total).
4. Adds the initial state word-by-word to the working copy (mod 2^32). This final addition prevents the mixing from being reversible.
5. Returns `serialize_state(working_copy)`.

In [ ]:
def double_round(state: list[int]) -> list[int]:
    """
    Apply one ChaCha20 double round (column round + diagonal round) in place.

    Modifies the state list in place AND returns it.

    Column round -- quarter round on each column of the 4x4 state:
    Diagonal round -- quarter round on each diagonal:

    Parameters
    ----------
    state : Mutable list of 16 32-bit words (modified in place).

    Returns
    -------
    The same list after applying both rounds (for convenience).
    """
    pass  # TODO


def chacha20_block(key: bytes, nonce: bytes, counter: int) -> bytes:
    """
    Generate one 64-byte ChaCha20 keystream block.

    Parameters
    ----------
    key     : 32-byte secret key.
    nonce   : 12-byte nonce.
    counter : 32-bit block counter.

    Returns
    -------
    64-byte keystream block.
    """
    pass  # TODO


# -- Tests (RFC 8439 ss.2.3.2) -------------------------------------------------

# double_round sanity: a non-trivial state must be altered
_nonzero = list(range(16))
_nonzero[0] = 0x61707865
double_round(_nonzero)
assert any(w != i for i, w in enumerate(_nonzero)), "double_round must alter the state"

# Full block test -- RFC 8439 ss.2.3.2
_key_block = bytes(range(32))
_nonce_block = bytes.fromhex("000000090000004a00000000")  # fmt:skip
_block = chacha20_block(_key_block, _nonce_block, 1)

assert len(_block) == 64, "Block must be exactly 64 bytes"

# Expected output words (little-endian) from RFC 8439 ss.2.3.2
# fmt:off
_expected_words = [
    0xe4e7f110, 0x15593bd1, 0x1fdd0f50, 0xc47120a3,
    0xc7f4d1c7, 0x0368c033, 0x9aaa2204, 0x4e6cd4c3,
    0x466482d2, 0x09aa9f07, 0x05d7c214, 0xa2028bd9,
    0xd19c12b5, 0xb94e16de, 0xe883d0cb, 0x4e3c50a2,
]
# fmt:on
_got_words = [int.from_bytes(_block[i * 4 : (i + 1) * 4], "little") for i in range(16)]
assert _got_words == _expected_words, "Block mismatch at word(s): " + str(
    [i for i in range(16) if _got_words[i] != _expected_words[i]]
)

print("Part C -- All tests passed.")

### Part D - Stream Cipher: `chacha20_encrypt` and `chacha20_decrypt`

With the block function in place, building the stream cipher is straightforward:

1. **Slice** the plaintext into 64-byte chunks.
2. For each chunk at index `i`, call `chacha20_block(key, nonce, counter + i)` to produce a 64-byte keystream block.
3. **XOR** the chunk with the keystream, byte by byte.
4. The **last chunk** may be shorter than 64 bytes,  only use as many keystream bytes as needed (do not pad the plaintext).

Because XOR is its own inverse, **encryption and decryption are the same operation**. `chacha20_decrypt` simply calls `chacha20_encrypt`.

> **Important:** the `counter` parameter is the *starting* counter value for the first block. Subsequent blocks use `counter + 1`, `counter + 2`, etc. RFC 8439 examples typically use `counter = 1` for data encryption.

> **Hint:** You can take advantage of `zip(chunk, keystream_block))`, that will produce (int, int) tuples, for XOR'ing the keystream with the plaintext (or ciphertext).

In [ ]:
def chacha20_encrypt(plaintext: bytes, key: bytes, nonce: bytes, counter: int) -> bytes:
    """
    Encrypt (or decrypt) an arbitrary-length message with ChaCha20.

    Splits plaintext into 64-byte blocks. For block i (0-indexed):
        keystream = chacha20_block(key, counter + i, nonce)
        ciphertext_chunk = bytes(p ^^ k for p, k in zip(plaintext_chunk, keystream))

    The last block may be shorter than 64 bytes; only the needed keystream
    bytes are used (never pad plaintext).

    Parameters
    ----------
    plaintext : Message of any length (bytes).
    key       : 32-byte secret key.
    nonce     : 12-byte nonce.
    counter   : Initial block counter (RFC 8439 uses 1 for data, 0 for Poly1305).

    Returns
    -------
    Ciphertext of the same length as plaintext.
    """
    # CRITICAL: Use ^^ for XOR in SageMath, NOT ^

    pass  # TODO


def chacha20_decrypt(ciphertext: bytes, key: bytes, nonce: bytes, counter: int) -> bytes:  # fmt:skip
    """
    Decrypt a ChaCha20 ciphertext.

    ChaCha20 is a stream cipher: decryption is identical to encryption.
    Delegate entirely to chacha20_encrypt.

    Parameters
    ----------
    ciphertext : Ciphertext of any length (bytes).
    key        : 32-byte secret key.
    nonce      : 12-byte nonce used during encryption.
    counter    : Initial block counter used during encryption.

    Returns
    -------
    Recovered plaintext.
    """
    pass  # TODO


# -- Tests (RFC 8439 ss.2.4.2) -------------------------------------------------

# Encrypting zeros must equal the raw keystream
_key_s = b"\x00" * 32
_n_s = b"\x00" * 12
_ct_z = chacha20_encrypt(b"\x00" * 64, _key_s, _n_s, 0)
assert _ct_z == chacha20_block(_key_s, _n_s, 0), (
    "Encrypting zeros must equal the raw keystream block"
)

# Round-trip: decrypt(encrypt(msg)) == msg
_msg = b"Hello, ChaCha20 from RFC 8439!"
_k_rt = bytes(range(32))
_n_rt = bytes(range(12))
assert (
    chacha20_decrypt(chacha20_encrypt(_msg, _k_rt, _n_rt, 0), _k_rt, _n_rt, 0) == _msg
), "Round-trip failed"

# RFC 8439 ss.2.4.2 full encryption test vector
_key_tv = bytes(range(32))
_nonce_tv = bytes.fromhex("000000000000004a00000000")  # fmt:skip
_pt_tv = (
    b"Ladies and Gentlemen of the class of '99: If I could offer you "
    b"only one tip for the future, sunscreen would be it."
)
_ct_tv = chacha20_encrypt(_pt_tv, _key_tv, _nonce_tv, 1)

# fmt:off
_expected_ct_hex = (
    "6e2e359a2568f98041ba0728dd0d6981"
    "e97e7aec1d4360c20a27afccfd9fae0b"
    "f91b65c5524193d4b5d2f0b4282227d4"
    "94d7e4b576c8e0e4a9d6a28b1815c930"
    "8a00cc36f0a4b3e3df49b71a20e34f14"
    "7cb7791059bbe8c05e0e0f694a5cb6ea"
    "39c01"
)
# fmt:on
assert _ct_tv.hex() == _expected_ct_hex, (
    f"RFC 8439 ss.2.4.2 ciphertext mismatch\nGot:      {_ct_tv.hex()}\nExpected: {_expected_ct_hex}"
)

assert chacha20_decrypt(_ct_tv, _key_tv, _nonce_tv, 1) == _pt_tv, (
    "Decryption of RFC 8439 test vector failed"
)

print("Part D -- All tests passed.")
print("Full ChaCha20 implementation verified against RFC 8439!")